# Notebook 03 — IBM Granite + RAG Decision Support

## CleanGanga — Prayagraj

This notebook continues from Notebook 02 and does not repeat EDA, hotspot calculations, or Logistic Regression training.

It loads the exported station/hotspot results, prepares grounded AI context, creates an IBM Granite prompt, provides an optional watsonx.ai connection, builds a transparent retrieval baseline, and combines retrieved evidence with station metrics.

**Core architecture:** Station evidence + verified knowledge → Retrieval → IBM Granite → Grounded explanation → Decision support.

Granite must not invent measurements, pollution sources, regulatory conclusions, or sources.

## 1. Load Notebook 02 Outputs

Notebook 02 should already have created `data/station_summary.csv` and `data/hotspot_ranking.csv`.

In [16]:
from pathlib import Path
import pandas as pd
import numpy as np
import json
import re
import os

DATA_DIR = Path("../data")
station_summary_path = DATA_DIR / "station_summary.csv"
hotspot_ranking_path = DATA_DIR / "hotspot_ranking.csv"

if not station_summary_path.exists():
    raise FileNotFoundError(f"Missing: {station_summary_path}")
if not hotspot_ranking_path.exists():
    raise FileNotFoundError(f"Missing: {hotspot_ranking_path}")

station_summary = pd.read_csv(station_summary_path)
hotspot_ranking = pd.read_csv(hotspot_ranking_path)

print("Station summary:", station_summary.shape)
print("Hotspot ranking:", hotspot_ranking.shape)

Station summary: (6, 28)
Hotspot ranking: (6, 28)


## 2. Inspect the AI Input

Check the exact exported columns before constructing prompts. This is validation, not a repeat of EDA.

In [17]:
print("Station summary columns:")
print(station_summary.columns.tolist())

print("\nHotspot ranking columns:")
print(hotspot_ranking.columns.tolist())

display(hotspot_ranking.head())

Station summary columns:
['Station', 'observations', 'mean_bod', 'max_bod', 'mean_fc', 'max_fc', 'bod_exceedances', 'fc_desirable_exceedances', 'fc_max_exceedances', 'mean_bod_exceedance', 'mean_fc_exceedance', 'latitude', 'longitude', 'polluted_observations', 'persistence', 'mean_bod_ratio', 'max_bod_ratio', 'mean_fc_ratio', 'max_fc_ratio', 'bod_trend', 'fc_trend', 'anomalous_observations', 'anomaly_rate', 'persistence_score', 'severity_raw', 'severity_score', 'anomaly_score', 'hotspot_score_baseline']

Hotspot ranking columns:
['Station', 'observations', 'mean_bod', 'max_bod', 'mean_fc', 'max_fc', 'bod_exceedances', 'fc_desirable_exceedances', 'fc_max_exceedances', 'mean_bod_exceedance', 'mean_fc_exceedance', 'latitude', 'longitude', 'polluted_observations', 'persistence', 'mean_bod_ratio', 'max_bod_ratio', 'mean_fc_ratio', 'max_fc_ratio', 'bod_trend', 'fc_trend', 'anomalous_observations', 'anomaly_rate', 'persistence_score', 'severity_raw', 'severity_score', 'anomaly_score', 'hotspo

,Station,observations,mean_bod,max_bod,mean_fc,max_fc,bod_exceedances,fc_desirable_exceedances,fc_max_exceedances,mean_bod_exceedance,...,max_fc_ratio,bod_trend,fc_trend,anomalous_observations,anomaly_rate,persistence_score,severity_raw,severity_score,anomaly_score,hotspot_score_baseline
0,GANGA AT KADAGHAT ALLAHABAD,22,2.659091,2.9,960.909091,1400.0,0,22,0,0.0,...,2.80,0.150000,-147.000000,2,0.090909,1.000000,0.921818,0.900672,0.50000,0.800224
1,GANGA AT ALLAHABAD D/S (SANGAM) U.P.,23,2.665217,2.9,1011.739130,1400.0,0,23,0,0.0,...,2.80,0.074286,-29.714286,1,0.043478,1.000000,1.023478,1.000000,0.23913,0.746377
2,GANGA AT ALLAHABAD (RASOOLABAD) U.P.,23,2.734783,2.9,1003.913043,1400.0,0,23,0,0.0,...,2.80,0.090000,-28.000000,1,0.043478,1.000000,1.007826,0.984707,0.23913,0.741279
3,YAMUNA AT ALLAHABAD D/S (BALUA GHAT) U.P,11,2.545455,2.7,654.545455,930.0,0,9,0,0.0,...,1.86,NaN,NaN,2,0.181818,0.818182,0.320000,0.312659,1.00000,0.710280
4,RIVER GANGA A/C TAMSA RIVER SIRSA SON BARSA,23,2.621739,2.8,792.173913,1100.0,0,23,0,0.0,...,2.20,0.110000,-86.000000,1,0.043478,1.000000,0.584348,0.570943,0.23913,0.603358


## 3. Build Structured Station Context

Only relevant station-level evidence is passed to the language model.

In [18]:
def first_existing(row, candidates, default=np.nan):
    for column in candidates:
        if column in row.index:
            return row[column]
    return default

def build_station_context(row):
    return {
        "station": first_existing(row, ["Station"]),
        "latitude": first_existing(row, ["latitude", "Latitude"]),
        "longitude": first_existing(row, ["longitude", "Longitude"]),
        "observations": first_existing(row, ["observations"]),
        "persistence": first_existing(row, ["persistence"]),
        "mean_bod": first_existing(row, ["mean_bod"]),
        "max_bod": first_existing(row, ["max_bod"]),
        "mean_fc": first_existing(row, ["mean_fc"]),
        "max_fc": first_existing(row, ["max_fc"]),
        "anomaly_rate": first_existing(row, ["anomaly_rate"]),
        "hotspot_score_baseline": first_existing(row, ["hotspot_score_baseline", "hotspot_score"])
    }

station_contexts = [build_station_context(row) for _, row in hotspot_ranking.iterrows()]

print("Contexts created:", len(station_contexts))
print(json.dumps(station_contexts[0], indent=2, default=str))

Contexts created: 6
{
  "station": "GANGA AT KADAGHAT ALLAHABAD",
  "latitude": 25.443124,
  "longitude": 81.887148,
  "observations": 22,
  "persistence": 1.0,
  "mean_bod": 2.659090909090909,
  "max_bod": 2.9,
  "mean_fc": 960.9090909090908,
  "max_fc": 1400.0,
  "anomaly_rate": 0.0909090909090909,
  "hotspot_score_baseline": 0.8002239901135398
}


## 4. Build a Grounded Granite Prompt

The prompt separates evidence from interpretation and restricts unsupported claims.

In [19]:
def format_value(value, digits=3):
    if pd.isna(value):
        return "not available"
    if isinstance(value, (float, np.floating)):
        return f"{value:.{digits}f}"
    return str(value)

def build_granite_prompt(context):
    station = context["station"]
    latitude = format_value(context["latitude"])
    longitude = format_value(context["longitude"])
    observations = format_value(context["observations"], 0)
    persistence = format_value(context["persistence"])
    mean_bod = format_value(context["mean_bod"])
    max_bod = format_value(context["max_bod"])
    mean_fc = format_value(context["mean_fc"])
    max_fc = format_value(context["max_fc"])
    anomaly_rate = format_value(context["anomaly_rate"])
    score = format_value(context["hotspot_score_baseline"])

    prompt = f'''
You are an environmental decision-support assistant.

Explain the monitoring evidence for this station.

Station: {station}
Latitude: {latitude}
Longitude: {longitude}
Number of observations: {observations}
Persistence: {persistence}
Mean BOD (mg/L): {mean_bod}
Maximum BOD (mg/L): {max_bod}
Mean Fecal Coliform (MPN/100mL): {mean_fc}
Maximum Fecal Coliform (MPN/100mL): {max_fc}
Anomaly rate: {anomaly_rate}
Baseline hotspot score: {score}

Tasks:
1. Summarize the strongest evidence in 2–4 sentences.
2. Explain why the station received its relative hotspot ranking.
3. Distinguish measured evidence from interpretation.
4. State that the hotspot score is a decision-support indicator, not proof of a pollution source or regulatory violation.
5. Mention important uncertainty or limitations.

Rules:
- Use only the values supplied above.
- Do not invent measurements.
- Do not claim a specific pollution source unless evidence is explicitly supplied.
- Do not claim causation.
- Do not present the ranking as a regulatory decision.
'''
    return prompt.strip()

example_prompt = build_granite_prompt(station_contexts[0])
print(example_prompt)

You are an environmental decision-support assistant.

Explain the monitoring evidence for this station.

Station: GANGA AT KADAGHAT ALLAHABAD
Latitude: 25.443
Longitude: 81.887
Number of observations: 22
Persistence: 1.000
Mean BOD (mg/L): 2.659
Maximum BOD (mg/L): 2.900
Mean Fecal Coliform (MPN/100mL): 960.909
Maximum Fecal Coliform (MPN/100mL): 1400.000
Anomaly rate: 0.091
Baseline hotspot score: 0.800

Tasks:
1. Summarize the strongest evidence in 2–4 sentences.
2. Explain why the station received its relative hotspot ranking.
3. Distinguish measured evidence from interpretation.
4. State that the hotspot score is a decision-support indicator, not proof of a pollution source or regulatory violation.
5. Mention important uncertainty or limitations.

Rules:
- Use only the values supplied above.
- Do not invent measurements.
- Do not claim a specific pollution source unless evidence is explicitly supplied.
- Do not claim causation.
- Do not present the ranking as a regulatory decision.

## 5. IBM watsonx.ai Credentials

Keep credentials outside the notebook.

Expected environment variables:
- `WATSONX_APIKEY`
- `WATSONX_PROJECT_ID`
- `WATSONX_URL` (optional)
- `GRANITE_MODEL_ID` (optional)

In [20]:
WATSONX_APIKEY = os.getenv("WATSONX_APIKEY")
WATSONX_PROJECT_ID = os.getenv("WATSONX_PROJECT_ID")
WATSONX_URL = os.getenv("WATSONX_URL", "https://us-south.ml.cloud.ibm.com")

IBM_GRANITE_READY = bool(WATSONX_APIKEY and WATSONX_PROJECT_ID)

print("IBM Granite credentials configured:", IBM_GRANITE_READY)

IBM Granite credentials configured: False


## 6. IBM watsonx.ai SDK

If the SDK is not installed, run the commented installation command once. Never put API keys in Git.

In [21]:
# If required, run once:
# %pip install -q ibm-watsonx-ai

if IBM_GRANITE_READY:
    try:
        from ibm_watsonx_ai import Credentials
        from ibm_watsonx_ai.foundation_models import ModelInference

        credentials = Credentials(
            url=WATSONX_URL,
            api_key=WATSONX_APIKEY
        )
        print("IBM watsonx.ai SDK imported.")
    except ImportError:
        print("SDK not installed. Uncomment the pip line above and run it once.")
else:
    print("Skipping IBM SDK import until credentials are configured.")

Skipping IBM SDK import until credentials are configured.


## 7. Granite Model Configuration

Model availability varies by IBM watsonx.ai account and region. Verify the model ID in your environment.

In [22]:
GRANITE_MODEL_ID = os.getenv(
    "GRANITE_MODEL_ID",
    "ibm/granite-3-3-8b-instruct"
)

print("Model ID:", GRANITE_MODEL_ID)

Model ID: ibm/granite-3-3-8b-instruct


## 8. Granite Inference Function

Live inference runs only when IBM credentials are configured.

In [23]:
def generate_granite_explanation(prompt):
    if not IBM_GRANITE_READY:
        return None

    try:
        model = ModelInference(
            model_id=GRANITE_MODEL_ID,
            credentials=credentials,
            project_id=WATSONX_PROJECT_ID,
            params={
                "max_new_tokens": 350,
                "temperature": 0.2
            }
        )
        return model.generate_text(prompt=prompt)

    except Exception as exc:
        print("Granite inference failed:", type(exc).__name__, exc)
        return None

granite_output = generate_granite_explanation(example_prompt)

if granite_output:
    print(granite_output)
else:
    print("No live Granite output. Prompt is ready for inference.")

No live Granite output. Prompt is ready for inference.


## 9. Create the RAG Knowledge Base Folder

Place verified `.txt` or `.md` environmental references in `data/knowledge_base/`.

Recommended sources include official water-quality standards/guidelines, monitoring documentation, project methodology, and verified definitions of BOD and Fecal Coliform.

In [24]:
KNOWLEDGE_DIR = DATA_DIR / "knowledge_base"
KNOWLEDGE_DIR.mkdir(parents=True, exist_ok=True)

knowledge_files = sorted(
    list(KNOWLEDGE_DIR.glob("*.txt")) +
    list(KNOWLEDGE_DIR.glob("*.md"))
)

print("Knowledge-base files:", len(knowledge_files))
for path in knowledge_files:
    print("-", path.name)

Knowledge-base files: 0


## 10. Chunk Knowledge Documents

We use a simple, inspectable chunking method as the first retrieval baseline.

In [25]:
def chunk_text(text, chunk_size=1200, overlap=200):
    text = re.sub(r"\s+", " ", text).strip()

    if not text:
        return []

    chunks = []
    start = 0

    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunks.append(text[start:end])

        if end == len(text):
            break

        start = end - overlap

    return chunks

knowledge_chunks = []

for path in knowledge_files:
    text = path.read_text(encoding="utf-8", errors="ignore")

    for chunk_id, chunk in enumerate(chunk_text(text)):
        knowledge_chunks.append({
            "source": path.name,
            "chunk_id": chunk_id,
            "text": chunk
        })

print("Knowledge chunks:", len(knowledge_chunks))

Knowledge chunks: 0


## 11. TF-IDF Retrieval Baseline

This transparent retrieval baseline can later be replaced by embeddings/vector search.

In [26]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

if knowledge_chunks:
    knowledge_texts = [item["text"] for item in knowledge_chunks]
    vectorizer = TfidfVectorizer(
        stop_words="english",
        ngram_range=(1, 2)
    )
    knowledge_matrix = vectorizer.fit_transform(knowledge_texts)
    print("TF-IDF index built.")
else:
    vectorizer = None
    knowledge_matrix = None
    print("No knowledge documents found yet.")

def retrieve(query, top_k=3):
    if vectorizer is None or knowledge_matrix is None:
        return []

    query_vector = vectorizer.transform([query])
    scores = cosine_similarity(query_vector, knowledge_matrix)[0]
    ranked_indices = np.argsort(scores)[::-1][:top_k]

    return [
        {
            **knowledge_chunks[index],
            "score": float(scores[index])
        }
        for index in ranked_indices
    ]

No knowledge documents found yet.


## 12. Test Retrieval

In [27]:
retrieval_query = "What do BOD and fecal coliform indicate in water quality monitoring?"
retrieved = retrieve(retrieval_query, top_k=3)

if not retrieved:
    print("No documents retrieved. Add verified .txt/.md files to data/knowledge_base/.")
else:
    for result in retrieved:
        print(
            f"Source: {result['source']} | "
            f"Chunk: {result['chunk_id']} | "
            f"Score: {result['score']:.3f}"
        )
        print(result["text"][:700])
        print("-" * 80)

No documents retrieved. Add verified .txt/.md files to data/knowledge_base/.


## 13. Build the RAG Prompt

The model receives project-computed station evidence plus retrieved reference evidence.

In [28]:
def build_rag_prompt(context, retrieved_documents, user_question):
    evidence_blocks = []

    for item in retrieved_documents:
        evidence_blocks.append(
            "[Source: " + str(item["source"]) +
            " | Chunk: " + str(item["chunk_id"]) + "]\n" +
            str(item["text"])
        )

    retrieved_evidence = "\n\n".join(evidence_blocks)
    if not retrieved_evidence:
        retrieved_evidence = "No external reference evidence was retrieved."

    station = context["station"]
    observations = format_value(context["observations"], 0)
    persistence = format_value(context["persistence"])
    mean_bod = format_value(context["mean_bod"])
    max_bod = format_value(context["max_bod"])
    mean_fc = format_value(context["mean_fc"])
    max_fc = format_value(context["max_fc"])
    anomaly_rate = format_value(context["anomaly_rate"])
    score = format_value(context["hotspot_score_baseline"])

    prompt = f'''
You are an environmental decision-support assistant.

User question:
{user_question}

STATION EVIDENCE
Station: {station}
Observations: {observations}
Persistence: {persistence}
Mean BOD (mg/L): {mean_bod}
Maximum BOD (mg/L): {max_bod}
Mean Fecal Coliform (MPN/100mL): {mean_fc}
Maximum Fecal Coliform (MPN/100mL): {max_fc}
Anomaly rate: {anomaly_rate}
Baseline hotspot score: {score}

RETRIEVED REFERENCE EVIDENCE
{retrieved_evidence}

Answer using only the supplied station evidence and retrieved reference evidence.

Rules:
- Do not invent measurements or sources.
- Name the retrieved source when using its information.
- Distinguish project measurements from reference information.
- Do not claim pollution source or causation without evidence.
- Do not turn the hotspot score into a regulatory judgment.
- If evidence is insufficient, say so.
'''
    return prompt.strip()

rag_prompt = build_rag_prompt(
    station_contexts[0],
    retrieved,
    "Why is this station considered a potential hotspot?"
)

print(rag_prompt)

You are an environmental decision-support assistant.

User question:
Why is this station considered a potential hotspot?

STATION EVIDENCE
Station: GANGA AT KADAGHAT ALLAHABAD
Observations: 22
Persistence: 1.000
Mean BOD (mg/L): 2.659
Maximum BOD (mg/L): 2.900
Mean Fecal Coliform (MPN/100mL): 960.909
Maximum Fecal Coliform (MPN/100mL): 1400.000
Anomaly rate: 0.091
Baseline hotspot score: 0.800

RETRIEVED REFERENCE EVIDENCE
No external reference evidence was retrieved.

Answer using only the supplied station evidence and retrieved reference evidence.

Rules:
- Do not invent measurements or sources.
- Name the retrieved source when using its information.
- Distinguish project measurements from reference information.
- Do not claim pollution source or causation without evidence.
- Do not turn the hotspot score into a regulatory judgment.
- If evidence is insufficient, say so.


## 14. RAG + Granite

Run live generation only after manually checking that retrieved evidence is relevant.

In [29]:
rag_granite_output = generate_granite_explanation(rag_prompt)

if rag_granite_output:
    print(rag_granite_output)
else:
    print("RAG prompt is ready, but live Granite inference is not configured.")

RAG prompt is ready, but live Granite inference is not configured.


## 15. Generate Explanations for Top Stations

During development, keep this to a few stations so outputs can be manually reviewed.

In [30]:
top_n = min(3, len(station_contexts))
generated_results = []

for context in station_contexts[:top_n]:
    query = (
        f"water quality interpretation for {context['station']} "
        "BOD fecal coliform monitoring"
    )

    docs = retrieve(query, top_k=3)

    prompt = build_rag_prompt(
        context,
        docs,
        "Explain the evidence behind this station's hotspot ranking."
    )

    output = generate_granite_explanation(prompt)

    generated_results.append({
        "station": context["station"],
        "hotspot_score": context["hotspot_score_baseline"],
        "retrieved_sources": [doc["source"] for doc in docs],
        "granite_output": output
    })

for item in generated_results:
    print("=" * 90)
    print("Station:", item["station"])
    print("Hotspot score:", item["hotspot_score"])
    print("Sources:", item["retrieved_sources"])
    print(item["granite_output"] or "[Granite not configured]")

Station: GANGA AT KADAGHAT ALLAHABAD
Hotspot score: 0.8002239901135398
Sources: []
[Granite not configured]
Station: GANGA AT ALLAHABAD D/S (SANGAM) U.P.
Hotspot score: 0.746376811594203
Sources: []
[Granite not configured]
Station: GANGA AT ALLAHABAD (RASOOLABAD) U.P.
Hotspot score: 0.7412791055619175
Sources: []
[Granite not configured]


## 16. AI Output Evaluation

For each generated explanation, check:

- **Grounding:** Are station numbers correct?
- **Hallucination:** Did it invent a source, measurement, date, or cause?
- **Attribution:** Are retrieved sources identified?
- **Uncertainty:** Are limitations communicated?
- **Usefulness:** Does the explanation help explain the ranking?

Record failures and improve retrieval/prompting rather than hiding them.

## 17. Evaluation Record

| Test | Result | Issue | Action |
|---|---|---|---|
| Station metrics reproduced correctly | | | |
| Retrieved evidence relevant | | | |
| Source attribution correct | | | |
| No unsupported causal claim | | | |
| Uncertainty communicated | | | |
| Explanation useful | | | |

## 18. Responsible AI and Limitations

- The hotspot score is a decision-support indicator, not a regulatory judgment.
- The Logistic Regression target is rule-derived, not independent scientific ground truth.
- Current BOD/Fecal Coliform measurements were excluded from the first ML feature set to reduce target leakage.
- Small and uneven monitoring data can limit generalization.
- Retrieval quality depends on the quality of the knowledge base.
- Granite can generate fluent but incorrect statements; important outputs must be grounded and reviewed.
- The system does not establish causation, identify responsible parties, or make regulatory decisions.

## 19. Final Architecture

```text
Monitoring Dataset
       ↓
Notebook 01
Cleaning + EDA + Assessment
       ↓
Notebook 02
Hotspot Analysis + ML Baseline
       ↓
Structured Station Evidence
       +
Verified Knowledge Base
       ↓
Retrieval
       ↓
IBM Granite
       ↓
Grounded Explanation
       ↓
Decision-Support Application
```

### Technology roles

- Python/Pandas → analysis
- Scikit-learn → ML and retrieval baseline
- IBM Granite → grounded natural-language generation
- RAG → evidence retrieval before generation
- IBM BOB → AI-assisted application development
- Git/GitHub → reproducibility and version control

## 20. Next Development Stage

After validating this notebook, use IBM BOB to build the user-facing application:

**Station selection → hotspot metrics → retrieved evidence → Granite explanation → references**

Document the architecture, experiments, evaluation, limitations, failed attempts, and reproducibility steps.

Never commit API keys or other secrets.